<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Appendix B · NumPy and pandas Reference

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Notebook Goals

This notebook turns Appendix B into runnable NumPy/pandas examples that you can
copy into your own projects.

### Getting Help

- Appendix B is the primary reference for the snippets here.
- Chapter 3 shows these patterns inside a full research workflow.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"


### Data Loading
We load the EOD panel once for the appendix examples.

In [ ]:
prices = pd.read_csv(DATA_PATH,
parse_dates=['Date']).set_index('Date').sort_index().ffill()

## 1. Array Creation and Broadcasting

We map basic linear algebra notation to NumPy arrays.

In [ ]:
w = np.array([0.3, 0.2, 0.5])
mu = np.array([0.08, 0.05, 0.06])
Sigma = np.array(
    [
        [0.10, 0.02, 0.01],
        [0.02, 0.05, 0.015],
        [0.01, 0.015, 0.04],
    ]
)
port_ret = w @ mu
port_var = w @ Sigma @ w
float(port_ret), float(np.sqrt(port_var))

## 2. Indexing and Slicing

We select assets and date ranges using label- and position-based indexing.

In [ ]:
subset = prices[['AAPL', 'SPY']].loc['2023-01-01': '2023-06-30']
subset.head()

## 3. Resampling and Grouping

Resampling helps move between daily, weekly, and monthly views.

In [ ]:
weekly = prices.resample('W-FRI').last()
monthly = prices.resample('ME').last()
weekly.head(), monthly.head()

## 4. Rolling Calculations

Rolling windows are the backbone of many feature-engineering routines.

In [ ]:
log_rets = np.log(prices / prices.shift(1)).dropna()
rolling_vol = log_rets['AAPL'].rolling(63).std() * np.sqrt(252)
fig, ax = plt.subplots(figsize=(12, 6))
rolling_vol.plot(ax=ax)
ax.set_title('AAPL 3M Rolling Volatility (annualized)')
ax.set_ylabel('Volatility')
plt.show()

## 5. Elementwise Operations and Boolean Masks

NumPy makes it easy to express elementwise transforms and conditional masks
without writing Python loops.


In [ ]:
vec = np.array([1.0, -2.0, 3.5, -4.2])
positive_mask = vec > 0
vec_squared = vec**2
vec[positive_mask], vec_squared

## 6. Pivoting and Stacking

pandas can reshape between wide and long formats; this is useful when
preparing cross-sectional panels for machine learning.


In [ ]:
wide = prices[['AAPL', 'SPY']].iloc[-5:]
long = wide.reset_index().melt(id_vars='Date', var_name='ticker', value_name='price')
long.head()

## 7. Exercises

1. Compute a 3-asset portfolio time series using vectorized operations.
2. Use <code>groupby</code> with a synthetic sector mapping to aggregate prices.
3. Build a helper that returns a tidy DataFrame of returns, volatilities, and Sharpe ratios.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">